(content:vq)=
# Lượng tử hóa vector (VQ)


Giả sử bạn đã ghi lại âm thanh tại nhiều vị trí khác nhau và muốn
phân loại chúng thành các nhóm tương tự. Nói cách khác, bạn có một
vector ngẫu nhiên $x$ mà bạn muốn mô tả bằng một
biểu diễn đơn giản. Ví dụ, các danh mục có thể tương ứng với văn phòng, đường phố,
hành lang và nhà ăn. Một phương pháp cổ điển cho nhiệm vụ này là chọn các
vector mẫu $c_{k}$, đại diện cho âm thanh điển hình trong mỗi
môi trường $k$. Để phân loại âm thanh, bạn tìm vector mẫu
gần nhất với bản ghi $x$. Dưới ký hiệu toán học,
bạn tìm $k^{^*}$ bằng

$$ k^* = \arg\min_k \|x-c_k\|^2. $$

Biểu thức trên tính toán sai số bình phương giữa $x$ và
mỗi vector $c_{k}$ và chọn chỉ số $k$ của
vector có sai số nhỏ nhất. Các vector $c_{k}$ khi đó
tạo thành một bảng mã và vector $x$ được lượng tử hóa thành
$c_{k^*}$. Đây là ý tưởng cơ bản đằng sau *lượng tử hóa vector,*
còn được gọi là *k-means*. 

Một minh họa về bảng mã vector đơn giản được hiển thị ở bên phải.
Dữ liệu đầu vào là một phân phối Gaussian được hiển thị bằng các chấm xám
và các vector mã $c_{k}$ bằng các vòng tròn đỏ. Với mỗi vector đầu vào,
ta tìm vector mã gần nhất và ranh giới giữa các vùng mà vector đầu vào
được gán cho một vector mã cụ thể
được minh họa bằng các đường màu xanh. Những vùng này được gọi là [vùng
Voronoi](https://en.wikipedia.org/wiki/Voronoi_diagram) và các đường
xanh là biên quyết định giữa các vector mã.

![vq.png](attachments/175511825.png) 

Ví dụ về bảng mã cho phân phối Gaussian 2D với 16 vector mã.

## Thước đo chất lượng bảng mã

Giả sử bạn có một tập hợp lớn các
vector $x_{h}$, và bạn muốn đánh giá mức độ mà bảng mã này
đại diện cho dữ liệu đầu vào. Kỳ vọng của sai số bình phương
xấp xỉ bằng trung bình trên toàn bộ dữ liệu, sao cho

$$ E_h\left[ \min_k \|x_h-c_k\|^2 \right] \approx \frac 1N
\sum_{h=1}^N \min_k \|x_h-c_k\|^2, $$

trong đó $E[ ]$ là toán tử kỳ vọng và $N$ là số lượng
vector đầu vào $x_{h}$. Ở trên, ta tìm vector mã
gần nhất với $x_{h}$, tính sai số bình phương của nó và lấy
kỳ vọng trên tất cả các đầu vào có thể. Điều này xấp xỉ bằng
trung bình của các sai số bình phương đó trên một tập hợp các vector đầu vào.

Để tìm tập hợp vector mã tối ưu $c_{k}$, ta cần
tối thiểu hóa sai số bình phương trung bình như sau

$$ \{c_k^*\} := \arg\min_{\{c_k\}}\, E_h\left[ \min_k
\|x_h-c_k\|^2 \right]  $$

hay cụ thể hơn, đối với một tập dữ liệu

$$ \{c_k^*\} := \arg\min_{\{c_k\}} \sum_{h=1}^N \min_k
\|x_h-c_k\|^2. $$

Thật không may, ta không có lời giải giải tích cho bài toán tối ưu
này, mà phải sử dụng các phương pháp số lặp.

## Tối ưu bảng mã

### Tối đại hóa kỳ vọng (EM)

Các phương pháp cổ điển để tìm bảng mã tối ưu là các biến thể của
tối đại hóa kỳ vọng (EM), dựa trên hai bước xen kẽ:

Thuật toán *Tối đại hóa Kỳ vọng (EM)*:

1.  Với mỗi vector $x_{h}$ trong một cơ sở dữ liệu lớn, tìm vector mã
    tốt nhất $c_{k}$.
2.  Với mỗi vector mã $c_{k}$;  
    1.  Tìm tất cả các vector $x_{h}$ được gán cho vector mã đó.
    2.  Tính trung bình của các vector đó.
    3.  Gán giá trị trung bình làm giá trị mới cho vector mã.
3.  Nếu đã hội tụ thì dừng, ngược lại quay lại bước 1.

Thuật toán này đảm bảo tạo ra một bảng mã tại mỗi bước mà
*không tệ hơn* bảng mã trước đó. Nghĩa là, mỗi vòng lặp sẽ
cải thiện cho đến khi tìm được một cực tiểu địa phương, tại đó nó dừng thay đổi.
Lý do là mỗi bước trong vòng lặp tìm một lời giải tốt nhất từng phần.
Trong bước đầu tiên, ta tìm các vector mã khớp nhất cho mỗi
vector dữ liệu $x_{h}$. Trong bước thứ hai, ta tính
trung bình trong từng cụm. Nghĩa là, giá trị trung bình mới chính xác hơn
vector mã trước đó ở chỗ nó làm giảm sai số bình phương trung bình. Nếu
trung bình bằng với vector mã trước đó, thì không có sự cải thiện nào. Chúng tôi đã chuẩn bị một đoạn mã mẫu về thuật toán này như sau:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import ipywidgets as widgets
from IPython.display import display

In [2]:
slider_dataset = widgets.SelectionSlider(
    options=['islands', 'moons', 'circles', 'swiss_roll'],
    value='islands',
    description='Dataset',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True)

slider_codebooks = widgets.SelectionSlider(
    options=[1, 2, 4, 8, 16, 32, 64, 128, 256, 512],
    value=8,
    description='No. codebook vectors',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True)

def on_value_change1(change):
    global dataset_name
    dataset_name = change['new']
slider_dataset.observe(on_value_change1, names='value')
display(slider_dataset)

def on_value_change2(change):
    global num_codebooks
    num_codebooks = change['new']
slider_codebooks.observe(on_value_change2, names='value')
display(slider_codebooks)

on_value_change1({'new':'islands'})
on_value_change2({'new':8})

SelectionSlider(continuous_update=False, description='Dataset', options=('islands', 'moons', 'circles', 'swiss…

SelectionSlider(continuous_update=False, description='No. codebook vectors', index=3, options=(1, 2, 4, 8, 16,…

In [3]:
data = np.load(f'./datasets/{dataset_name}.npy')
num_iterations = 15
data_dim = data.shape[1]

permuted_indices = np.random.permutation(data.shape[0])
codebooks = data[permuted_indices[0:num_codebooks]]
print('codebook vectors = ', codebooks.shape[0])

expanded_data = np.expand_dims(data, axis=2)
codebook_list = []

for iter in range(num_iterations):
        new_codebooks = codebooks.copy()
        codebook_list.append(new_codebooks)
        expanded_codebooks = np.expand_dims(codebooks.T, axis=0)
        euclidean_distances = np.sum(np.square(expanded_data - expanded_codebooks), axis=1)
        best_codebooks_indices = np.argmin(euclidean_distances, axis=1)

        for i in range(codebooks.shape[0]): # codebook update step
            indices = np.where(best_codebooks_indices == i)[0]
            mean_assigned_codebooks = np.mean(data[indices], axis=0, keepdims=True)
            codebooks[i] = mean_assigned_codebooks.copy()

fig, ax = plt.subplots()
line1 = ax.scatter([],[])
line2 = ax.scatter([],[],marker='X', c='red')
ax.set_xlim(data[:,0].min()-1, data[:,0].max()+1)
ax.set_ylim(data[:,1].min()-1, data[:,1].max()+1)
plt.close()

def animate(frame_num):
    line1.set_offsets(data)
    line2.set_offsets(codebook_list[frame_num])
    return (line1, line2)

anim = FuncAnimation(fig, animate, frames=len(codebook_list), interval=500)

from IPython.display import HTML
HTML(anim.to_jshtml())

codebook vectors =  8


Như đã đề cập ở trên, thuật toán này là nền tảng cho hầu hết các thuật toán
tối ưu bảng mã trong lượng tử hóa vector.
Có nhiều lý do khiến thuật toán đơn giản này thường không đủ hiệu quả khi đứng riêng.
Quan trọng nhất, thuật toán trên hội tụ chậm đến một nghiệm ổn định *và* nó thường
tìm được cực tiểu địa phương thay vì cực tiểu toàn cục.

Để cải thiện hiệu suất, ta có thể áp dụng một số phương pháp heuristic. Ví dụ,
ta có thể bắt đầu với một bảng mã nhỏ $ \{ c_k \}_{k=1}^K $
gồm $K$ phần tử và tối ưu nó bằng thuật toán EM. Sau đó, ta tách đôi
bảng mã bằng cách thêm một độ lệch nhỏ $d$, sao cho $
\|d\|<\epsilon $ và tạo bảng mã mới $ \{ \hat c_k
\}_{k=1}^{2K} := \{ c_k,\, c_k+d \}_{k=1}^K $ gồm 2$K$ phần tử.
Sau đó, ta chạy lại thuật toán EM trên bảng mã mới. Bảng mã do đó
tăng gấp đôi kích thước tại mỗi vòng lặp và ta tiếp tục cho đến khi đạt được
kích thước bảng mã mong muốn. Dưới đây là một đoạn mã mẫu cho phương pháp này.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import ipywidgets as widgets
from IPython.display import display

In [5]:
slider_dataset = widgets.SelectionSlider(
    options=['islands', 'moons', 'circles', 'swiss_roll'],
    value='islands',
    description='Dataset',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True)

slider_init_codebooks = widgets.SelectionSlider(
    options=[1, 2, 4, 8, 16, 32, 64, 128, 256],
    value=1,
    description='Init codebook vectors',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True)

slider_desired_codebooks = widgets.SelectionSlider(
    options=[2, 4, 8, 16, 32, 64, 128, 256, 512],
    value=8,
    description='Desired codebook vectors',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True)

def on_value_change1(change):
    global dataset_name
    dataset_name = change['new']
slider_dataset.observe(on_value_change1, names='value')
display(slider_dataset)

def on_value_change2(change):
    global initial_num_codebooks
    initial_num_codebooks = change['new']
slider_init_codebooks.observe(on_value_change2, names='value')
display(slider_init_codebooks)

def on_value_change3(change):
    global desired_num_codebooks
    desired_num_codebooks = change['new']
slider_desired_codebooks.observe(on_value_change3, names='value')
display(slider_desired_codebooks)

on_value_change1({'new':'islands'})
on_value_change2({'new':1})
on_value_change3({'new':8})

SelectionSlider(continuous_update=False, description='Dataset', options=('islands', 'moons', 'circles', 'swiss…

SelectionSlider(continuous_update=False, description='Init codebook vectors', options=(1, 2, 4, 8, 16, 32, 64,…

SelectionSlider(continuous_update=False, description='Desired codebook vectors', index=2, options=(2, 4, 8, 16…

In [6]:
if desired_num_codebooks <= initial_num_codebooks:
    raise ValueError("Initial No. of codebook vectors must be smaller than desired No. of codebook vectors")

In [7]:
data = np.load(f'./datasets/{dataset_name}.npy')
num_iterations = 8
data_dim = data.shape[1]
delta = 1e-2
expansion_times = int(np.round(np.log2(desired_num_codebooks) - np.log2(initial_num_codebooks)))

permuted_indices = np.random.permutation(data.shape[0])
codebooks = data[permuted_indices[0:initial_num_codebooks]]
print('initial codebook vectors = ', initial_num_codebooks)
print('desired codebook vectors = ', desired_num_codebooks)

data_reshaped = np.expand_dims(data, axis=2)
codebook_list = []

for expansion_idx in range(expansion_times+1):

    for iter in range(num_iterations):
        new_codebooks = codebooks.copy()
        codebook_list.append(new_codebooks)
        codebooks_reshaped = np.expand_dims(codebooks.T, axis=0)
        euclidean_distances = np.sum(np.square(data_reshaped - codebooks_reshaped), axis=1)
        best_codebooks_indices = np.argmin(euclidean_distances, axis=1)

        for i in range(codebooks.shape[0]): # codebook update step
            indices = np.where(best_codebooks_indices == i)[0]
            mean_assigned_codebooks = np.mean(data[indices], axis=0, keepdims=True)
            codebooks[i] = mean_assigned_codebooks.copy()

    expanded_codebooks = np.concatenate((codebooks, codebooks + delta), axis=0)
    codebooks = expanded_codebooks.copy()

final_codebook_list = [codebook_list[0]] + [codebook_list[num_iterations-1]] + codebook_list[num_iterations:]

fig, ax = plt.subplots()
line1 = ax.scatter([],[])
line2 = ax.scatter([],[],marker='X', c='red')
ax.set_xlim(data[:,0].min()-1, data[:,0].max()+1)
ax.set_ylim(data[:,1].min()-1, data[:,1].max()+1)
plt.close()

def animate(frame_num):
    line1.set_offsets(data)
    line2.set_offsets(final_codebook_list[frame_num])
    return (line1, line2)

anim = FuncAnimation(fig, animate, frames=len(final_codebook_list), interval=500)

from IPython.display import HTML
HTML(anim.to_jshtml())

initial codebook vectors =  1
desired codebook vectors =  8


Ưu điểm của phương pháp này là nó tập trung sự chú ý vào phần lớn
các điểm dữ liệu $x_{k}$, và bỏ qua các điểm ngoại lai. Kết quả
dự kiến sẽ ổn định hơn và khả năng hội tụ đến
cực tiểu địa phương nhỏ hơn. Nhược điểm là với phương pháp này, việc
tìm các cụm nhỏ tách biệt trở nên khó khăn hơn. Đó là vì
bảng mã ban đầu nằm gần trung tâm của toàn bộ khối dữ liệu,
việc thêm một độ lệch nhỏ vào các vector mã giữ các vector mã mới
gần trung tâm khối. 

Ngược lại, ta có thể bắt đầu với một bảng mã lớn, chẳng hạn coi toàn bộ
cơ sở dữ liệu đầu vào $x_{k}$ như một bảng mã. Sau đó, ta lặp lại việc
ghép từng cặp điểm gần nhau, cho đến khi bảng mã
được giảm xuống kích thước mong muốn. Không cần nói, đây sẽ là một quá trình
chậm nếu cơ sở dữ liệu lớn, nhưng sẽ rất hiệu quả trong việc tìm
các cụm điểm tách biệt. Dưới đây là một đoạn mã mẫu cho phương pháp này.

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import ipywidgets as widgets
from IPython.display import display

In [9]:
slider_dataset = widgets.SelectionSlider(
    options=['islands', 'moons', 'circles', 'swiss_roll'],
    value='islands',
    description='Dataset',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True)

slider_init_codebooks = widgets.SelectionSlider(
    options=[2, 4, 8, 16, 32, 64, 128, 256, 512],
    value=64,
    description='Init codebook vectors',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True)

slider_desired_codebooks = widgets.SelectionSlider(
    options=[1, 2, 4, 8, 16, 32, 64, 128, 256],
    value=4,
    description='Desired codebook vectors',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True)

def on_value_change1(change):
    global dataset_name
    dataset_name = change['new']
slider_dataset.observe(on_value_change1, names='value')
display(slider_dataset)

def on_value_change2(change):
    global initial_num_codebooks
    initial_num_codebooks = change['new']
slider_init_codebooks.observe(on_value_change2, names='value')
display(slider_init_codebooks)

def on_value_change3(change):
    global desired_num_codebooks
    desired_num_codebooks = change['new']
slider_desired_codebooks.observe(on_value_change3, names='value')
display(slider_desired_codebooks)

def merging_codebooks(codebooks):
    codebooks1 = np.expand_dims(codebooks, axis=2)
    codebooks2 = np.expand_dims(codebooks.T, axis=0)

    distances = np.sum(np.square(codebooks1 - codebooks2), axis=1)
    distances[distances == 0] = np.inf
    distances = distances.reshape(-1, 1)

    sorted_indcies = np.argsort(distances, axis=0)
    merged_codebooks = np.zeros((codebooks.shape[0] // 2, data_dim))
    selected_codebooks_indices = []
    counter = 0

    for j in range(sorted_indcies.shape[0]):
        row_idx = np.floor(sorted_indcies[j] / codebooks.shape[0]).astype(np.int64)[0]
        column_idx = np.remainder(sorted_indcies[j], codebooks.shape[0])[0]

        if (row_idx in selected_codebooks_indices) or (column_idx in selected_codebooks_indices):
            pass
        else:
            selected_codebooks_indices.extend((row_idx, column_idx))
            merged_codebooks[counter] = codebooks[row_idx] # defines new codebooks which are exactly data points
            counter += 1

        if counter == codebooks.shape[0] // 2:
            break

    return merged_codebooks

on_value_change1({'new':'islands'})
on_value_change2({'new':64})
on_value_change3({'new':8})

SelectionSlider(continuous_update=False, description='Dataset', options=('islands', 'moons', 'circles', 'swiss…

SelectionSlider(continuous_update=False, description='Init codebook vectors', index=5, options=(2, 4, 8, 16, 3…

SelectionSlider(continuous_update=False, description='Desired codebook vectors', index=2, options=(1, 2, 4, 8,…

In [10]:
if desired_num_codebooks >= initial_num_codebooks:
    raise ValueError("Desired No. of codebook vectors must be smaller than initial No. of codebook vectors")

In [11]:
data = np.load(f'./datasets/{dataset_name}.npy')
num_data_samples = data.shape[0]
num_iterations = 10
data_dim = data.shape[1]
delta = 1e-2

merging_times = int(np.round(np.log2(initial_num_codebooks) - np.log2(desired_num_codebooks)))

splitted_data = np.array_split(data, initial_num_codebooks)

codebooks = np.zeros((initial_num_codebooks, data_dim), dtype=np.float64)
for i in range(initial_num_codebooks):
    codebooks[i] = np.mean(splitted_data[i], axis=0)
print('initial codebook vectors = ', initial_num_codebooks)
print('desired codebook vectors = ', desired_num_codebooks)

data_reshaped = np.expand_dims(data, axis=2)
codebook_list = []

for merge_idx in range(merging_times+1):

    for iter in range(num_iterations):
        new_codebooks = codebooks.copy()
        codebook_list.append(new_codebooks)
        reshaped_codebooks = np.expand_dims(codebooks.T, axis=0)

        euclidean_distances = np.sum(np.square(data_reshaped - reshaped_codebooks), axis=1)

        best_codebooks_indices = np.argmin(euclidean_distances, axis=1)
        unique_best_indices = np.unique(best_codebooks_indices)

        for i in range(codebooks.shape[0]):
            indices = np.where(best_codebooks_indices == i)[0]
            if indices.size == 0:
                random_idx = np.random.choice(unique_best_indices)
                codebooks[i] = codebooks[random_idx] + delta
            else:
                mean_assigned_codebooks = np.mean(data[indices], axis=0, keepdims=True)
                codebooks[i] = mean_assigned_codebooks.copy()

    merged_codebooks = merging_codebooks(codebooks)
    codebooks = merged_codebooks.copy()

    if codebooks.shape[0] < desired_num_codebooks:
        break

fig, ax = plt.subplots()
line1 = ax.scatter([],[])
line2 = ax.scatter([],[],marker='X', c='red')
ax.set_xlim(data[:,0].min()-1, data[:,0].max()+1)
ax.set_ylim(data[:,1].min()-1, data[:,1].max()+1)
plt.close()

def animate(frame_num):
    line1.set_offsets(data)
    line2.set_offsets(codebook_list[frame_num])
    return (line1, line2)

anim = FuncAnimation(fig, animate, frames=len(codebook_list), interval=500)

from IPython.display import HTML
HTML(anim.to_jshtml())

initial codebook vectors =  64
desired codebook vectors =  8


Trong mọi trường hợp, việc tối ưu bảng mã vector là một nhiệm vụ khó khăn và ta
không có thuật toán thực tiễn nào đảm bảo tìm được
cực tiểu toàn cục. Giống như nhiều bài toán học máy khác, việc tối ưu
bảng mã phụ thuộc rất nhiều vào việc hiểu rõ dữ liệu của bạn. Bạn nên
sử dụng một thuật toán trước, sau đó phân tích đầu ra để tìm ra những gì có thể
cải thiện, và tiếp tục lặp lại quá trình tối ưu và phân tích này
cho đến khi đầu ra đủ tốt.

### Tối ưu với các nền tảng học máy

Một phương pháp hiện đại để mô hình hóa là [học máy](content:nn), trong đó các hiện tượng phức tạp được mô hình hóa bằng mạng nơ-ron. Thông thường, chúng được huấn luyện bằng các phương pháp kiểu [gradient-descent](https://en.wikipedia.org/wiki/Gradient_descent), trong đó các tham số được điều chỉnh lặp lại theo hướng cực tiểu, bằng cách đi theo gradient dốc nhất. Vì các gradient như vậy có thể được tự động tính toán trên các nền tảng học máy (sử dụng [quy tắc dây chuyền](https://en.wikipedia.org/wiki/Chain_rule)), chúng có thể được áp dụng cho các mô hình rất phức tạp. Do đó, chúng đã trở nên rất phổ biến và thành công. 

Cùng một kiểu huấn luyện như vậy cũng có thể được áp dụng dễ dàng cho các bộ lượng tử hóa vector. Tuy nhiên, có một vấn đề thực tiễn với phương pháp này. Việc ước lượng gradient của các tham số bằng quy tắc dây chuyền yêu cầu *tất cả* các gradient trung gian phải khác không. Tuy nhiên, các bộ lượng tử hóa lại có tính chất hằng từng đoạn nên gradient của chúng đều bằng không, do đó vô hiệu hóa quy tắc dây chuyền và gradient descent cho tất cả các tham số nằm sau bộ lượng tử hóa trong đồ thị tính toán. Một giải pháp thực tiễn được gọi là bộ ước lượng truyền thẳng (straight through estimator), trong đó các gradient được truyền nguyên vẹn qua bộ lượng tử hóa. Xấp xỉ này đơn giản để triển khai và thường mang lại hiệu suất đầy đủ. Dưới đây, chúng tôi đã chuẩn bị một đoạn mã mẫu để tối ưu bảng mã của bộ lượng tử hóa vector bằng thư viện học máy PyTorch.

Mặc dù phương pháp này hội tụ chậm hơn thuật toán EM, nó thường có lợi vì cho phép tối ưu toàn bộ hệ thống theo một mục tiêu duy nhất. Nghĩa là, nếu ta có nhiều mô-đun trong một hệ thống, sẽ có lợi nếu tương tác chung của chúng được xem xét trong quá trình tối ưu. Nếu các mô-đun được tối ưu độc lập, rất khó để dự đoán tất cả các tương tác và hiệu suất hệ thống có thể còn xa mức tối ưu.

## Độ phức tạp thuật toán

Lượng tử hóa vector có thể xử lý được với các bảng mã tương đối nhỏ,
chẳng hạn $K=32$ vector mã. Điều đó tương ứng với 5 bit thông tin. Đối với
nhiều ứng dụng, điều đó không cho độ chính xác đầy đủ - sai số bình phương
trung bình quá lớn. Ví dụ, các mô hình dự báo tuyến tính trong
mã hóa tiếng nói có thể được lượng tử hóa với 30 bit, tương ứng với $
K=2^{30}\approx 10^9 $ vector mã. Để tìm vector mã tốt nhất cho một
vector $x$ có độ dài $N=16$, ta cần tính
khoảng cách giữa mỗi vector mã và $x$, tương đương khoảng
$ 16\times10^9= 1.6\times10^{10} $ phép tính. Điều này
là bất khả thi trong các ứng dụng trực tuyến trên thiết bị di động. Thay vào đó, ta
cần tìm một phương pháp đơn giản hơn mà vẫn giữ được những ưu điểm của
thuật toán, nhưng giảm độ phức tạp tính toán.

Một phương pháp heuristic là sử dụng các bảng mã liên tiếp, trong đó tại mỗi
vòng lặp, ta lượng tử hóa sai số của vòng lặp trước. Nghĩa là, giả sử
ở vòng lặp đầu tiên ta có 8 bit, tương ứng với một
bảng mã $c_{k}$ gồm $K=256$ vector. Ta tìm vector mã khớp nhất
$c_{k^*}$ và tính phần dư $
x':=x-c_{k^*} $ . Ở giai đoạn thứ hai, ta sẽ tìm vector khớp nhất
cho $x'$ từ một bảng mã thứ hai $c_{k}'$. Ta có thể
thêm bao nhiêu tầng bảng mã tùy ý cho đến khi đạt được số bit
mong muốn. Phương pháp này được gọi là *lượng tử hóa vector đa tầng*.

Trong khi lượng tử hóa vector thông thường có thể tìm được nghiệm tối ưu, lượng tử hóa vector phân tách thường không cho nghiệm tối ưu toàn cục. Tuy nhiên, nó cho
các lời giải tốt, với độ phức tạp thuật toán
thấp hơn rất nhiều so với lượng tử hóa vector thông thường. Ví dụ, trong
ví dụ trên về 30 bit, ta có thể gán ba tầng bảng mã liên tiếp,
mỗi tầng 10 bit / $K=1024$, sao cho độ phức tạp tổng thể
là $ 3\times 16\times 2^{10} \approx 5\times10^4, $ mang lại
cải thiện với hệ số $ 3.5\times10^5. $ Với điều kiện
giảm độ chính xác có thể kiểm soát được, đây là một cải thiện đáng kể về
độ phức tạp.



## Ứng dụng

Có lẽ ứng dụng quan trọng nhất mà lượng tử hóa vector
được sử dụng trong xử lý tiếng nói là [mã hóa tiếng nói](content:telecom) với [Dự báo tuyến tính kích thích mã (CELP)](content:CELP), trong đó

-   [các hệ số dự báo tuyến tính (LPC)](content:linearprediction) được
    chuyển đổi thành tần số phổ tuyến (LSFs), thường được
    mã hóa bằng bộ lượng tử hóa vector đa tầng. 
-   các hệ số khuếh đại (năng lượng tín hiệu) của phần dư và dự báo dài hạn được
    mã hóa đồng thời bằng một bộ lượng tử hóa vector đơn tầng.

Các ứng dụng điển hình khác bao gồm

-   Trong tối ưu [mô hình hỗn hợp Gaussian (GMMs)](content:gmm),
    lượng tử hóa vector hữu ích để tìm ước lượng ban đầu của trung bình mỗi hỗn hợp.



## Thảo luận

Ưu điểm của lượng tử hóa vector là nó là một thuật toán đơn giản
cho độ chính xác cao. Thực tế, đối với việc lượng tử hóa dữ liệu phức tạp,
lượng tử hóa vector (về lý thuyết) là tối ưu trong các ứng dụng mã hóa tốc độ cố định.
Nó đơn giản ở chỗ một kỹ sư có kinh nghiệm có thể
triển khai nó trong vài giờ. Các nhược điểm của lượng tử hóa vector
bao gồm

-   Độ phức tạp; để lượng tử hóa chính xác, bạn cần các bảng mã lớn đến mức
    không thực tế. Do đó, phương pháp này không mở rộng tốt cho các bài toán
    lớn.
-   Tối ưu khó;  
    -   Dữ liệu huấn luyện; Lượng dữ liệu cần thiết để tối ưu một bảng mã vector
        là rất lớn. Mỗi vector mã phải được gán cho một
        số lượng lớn các vector dữ liệu, sao cho việc tính trung bình
        (trong thuật toán EM) có ý nghĩa.
    -   Hội tụ; ta không có đảm bảo rằng các thuật toán tối ưu
        tìm được cực tiểu toàn cục và ta không có đảm bảo rằng các cực tiểu
        địa phương là "đủ tốt".
-   Thiếu linh hoạt; bảng mã có kích thước cố định. Nếu ta muốn
    sử dụng các bảng mã có kích thước khác nhau, ví dụ, nếu ta muốn
    truyền dữ liệu với tốc độ bit thay đổi, thì ta phải tối ưu và
    lưu trữ một bảng mã lớn cho *mỗi tốc độ bit có thể*.
-   Thiếu nhận thức về cấu trúc nội tại; mô hình này mô tả dữ liệu bằng một
    bảng mã, mà không có bất kỳ hiểu biết sâu sắc nào về dữ liệu
    trong từng danh mục. Ví dụ, giả sử ta có hai lớp,
    tiếng nói và phi tiếng nói. Ngay cả khi tiếng nói rất linh hoạt,
    lớp phi tiếng nói lớn hơn rất nhiều. Tiếng nói là một tập con rất nhỏ
    của tất cả các âm thanh có thể. Do đó, phương sai trong lớp
    sẽ lớn hơn nhiều ở lớp phi tiếng nói. Hệ quả là, độ chính xác trong
    lớp phi tiếng nói sẽ thấp hơn nhiều.  
    Do đó, ta sẽ muốn tăng số lượng
    vector mã sao cho có độ chính xác đồng đều ở cả hai lớp. Nhưng
    khi đó ta mất đi sự tương ứng giữa các vector mã và
    các mô tả tự nhiên của tín hiệu.